In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.services.regression_model import MusicPopularityRegressor

Carregar dataset completo

In [7]:
df = pd.read_csv(project_root / "data" / "processed" / "dataset_clean.csv")
print(f"Dataset: {len(df)} musicas")

regressor = MusicPopularityRegressor(model_dir=project_root / "models")

FEATURES = ["danceability", "energy", "loudness_norm", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

print(f"\n Features usadas pela IA: {FEATURES}")

Dataset: 89740 musicas

 Features usadas pela IA: ['danceability', 'energy', 'loudness_norm', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


In [8]:
np.random.seed(42)
sample = df.sample(15).reset_index(drop=True)

display(sample[["track_name", "artists", "track_genre"] + FEATURES])

,track_name,artists,track_genre,danceability,energy,loudness_norm,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,Feliz Cumpleaños Ferxxo,Feid,latin,0.865,0.573,0.814420,0.0678,0.08630,0.000000,0.3050,0.563,94.999
1,Kamikadze wróć!,Papa Dance,synth-pop,0.786,0.866,0.735716,0.0399,0.28600,0.000000,0.2450,0.967,129.212
2,Do It Again,Steely Dan,country,0.695,0.583,0.763295,0.0303,0.15700,0.000076,0.0576,0.960,124.610
3,Romaria - Remasterizado,Renato Teixeira,mpb,0.505,0.346,0.725653,0.0263,0.60000,0.000003,0.0787,0.377,92.412
4,Oblivion,Esteban Morgado,tango,0.366,0.465,0.829514,0.0296,0.62600,0.004160,0.1610,0.353,116.530
5,What Am I Worth,George Jones,honky-tonk,0.577,0.712,0.779276,0.0500,0.57200,0.000000,0.1070,0.964,89.380
6,Purpurina,Alberto Gambino,spanish,0.890,0.643,0.803470,0.0688,0.13500,0.000000,0.2410,0.970,132.080
7,Four to the Floor,Starsailor,piano,0.530,0.782,0.789061,0.0259,0.02650,0.000187,0.0688,0.719,93.106
8,A Ella,KAROL G,reggae,0.730,0.769,0.824871,0.1780,0.24400,0.000021,0.1360,0.594,181.988
9,Her Or Me,Claude-Michel Schönberg;Claire Moore,show-tunes,0.222,0.131,0.642491,0.0376,0.81300,0.000025,0.1760,0.158,182.168


In [10]:
# a IA só verifica as features
x_sample = sample[FEATURES]

# previsão 
result = regressor.predict(x_sample)

# tabela de comparação
comparacao = pd.DataFrame({
    "Música": sample["track_name"].str[:35],
    "Artista": sample["artists"].str[:25],
    "Gênero": sample["track_genre"],
    "Real": sample["popularity"],
    "Previsto": result["predictions"],
    "Interpretação": result["interpretations"]
})

# calcular erro
comparacao["Erro"] = (comparacao["Real"] - comparacao["Previsto"]).abs().round(1)
comparacao["Acertou"] = comparacao["Erro"].apply(
    lambda e : "Sim!" if e <= 10 else ("Perto" if e <= 20 else "Errou")
)

print("Resultado IA vs Realidade:")
print("=" * 80)
display(comparacao)

Resultado IA vs Realidade:


,Música,Artista,Gênero,Real,Previsto,Interpretação,Erro,Acertou
0,Feliz Cumpleaños Ferxxo,Feid,latin,0,35.2,Baixa Popularidade,35.2,Errou
1,Kamikadze wróć!,Papa Dance,synth-pop,30,29.4,Baixa Popularidade,0.6,Sim!
2,Do It Again,Steely Dan,country,0,31.0,Baixa Popularidade,31.0,Errou
3,Romaria - Remasterizado,Renato Teixeira,mpb,47,36.4,Baixa Popularidade,10.6,Perto
4,Oblivion,Esteban Morgado,tango,16,34.1,Baixa Popularidade,18.1,Perto
5,What Am I Worth,George Jones,honky-tonk,12,28.4,Baixa Popularidade,16.4,Perto
6,Purpurina,Alberto Gambino,spanish,59,31.0,Baixa Popularidade,28.0,Errou
7,Four to the Floor,Starsailor,piano,2,34.7,Baixa Popularidade,32.7,Errou
8,A Ella,KAROL G,reggae,0,34.9,Baixa Popularidade,34.9,Errou
9,Her Or Me,Claude-Michel Schönberg;C,show-tunes,20,25.5,Baixa Popularidade,5.5,Sim!


In [ ]:
erro_medio = comparacao["Erro"].mean()
acertos_10 = (comparacao["Erro"] <= 10).sum()
perto_20 = (comparacao["Erro"] <= 20).sum()
total = len(comparacao)

print(f"Resultado FINAL")
print(f"{'='*50}")
print(f"Erro médio: {erro_medio:.1f} pontos (de 0 a 100)")
